# S4_0 — Stage 4: Master Execution Notebook
Domain pretraining of the Leaf-JEPA encoder on PlantVillage imagery.

## Pre-run Checklist
1. Copy `NORM_MEAN`/`NORM_STD` from `outputs/preprocessing/normalisation_stats.json` into `config_stage4.py`
2. Confirm `IJEPA_CHECKPOINT` points to the downloaded Meta checkpoint
3. Set `WANDB_ENTITY` in `config_stage4.py`
4. Set `PT_EPOCHS` (50 = minimum viable, 150 = recommended)
5. Confirm PlantDoc is NOT in any training loader

## Execution Order
| Notebook | Purpose | Stage 4 Required? |
|---|---|---|
| S4_1_setup_and_verify.ipynb | Environment + architecture verification | No |
| S4_2_masking_strategy.ipynb | Masking visualisation + saliency exploration | No |
| S4_3_pretrain_main.ipynb | **MAIN** — full pretraining loop | No (is Stage 4) |
| S4_4_lp_monitoring.ipynb | LP monitor analysis + convergence plots | After S4_3 partial |
| S4_5_masking_ablation.ipynb | Ablation: biased vs uniform masking | After S4_3 |
| S4_6_checkpoint_export.ipynb | Export Leaf-JEPA encoder + update config | After S4_3 |


In [ ]:
# ── Cell 1: Config check ──────────────────────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

from stage4_leaf_jepa_pretraining.config_stage4 import verify_config, PRETRAIN_DIR, PRETRAIN_CKPT_DIR, FIGURES_DIR
from stage4_leaf_jepa_pretraining.pretrain_utils import get_device  # if importing from baseline_utils, adjust path

PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)
PRETRAIN_CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

verify_config()


In [ ]:
# ── Cell 2: Post-run validation (run after all S4 notebooks complete) ─────────
import torch
from pathlib import Path
from stage4_leaf_jepa_pretraining.config_stage4 import LEAF_JEPA_CHECKPOINT, PRETRAIN_DIR, FIGURES_DIR, VIT_MODEL_NAME

print("POST-RUN VALIDATION — Stage 4")
print("="*55)

checks = {
    "Leaf-JEPA checkpoint exported":    LEAF_JEPA_CHECKPOINT,
    "Pretraining history JSON":          PRETRAIN_DIR / "pretrain_history.json",
    "LP monitor history JSON":           PRETRAIN_DIR / "lp_monitor_history.json",
    "Pretraining curves figure":         FIGURES_DIR  / "S4_pretrain_curves.png",
    "Masking visualisation":             FIGURES_DIR  / "S4_masking_examples.png",
    "Saliency map figure":               FIGURES_DIR  / "S4_saliency_examples.png",
    "Ablation results JSON":             PRETRAIN_DIR / "S4_masking_ablation.json",
}

passed = 0
for name, path in checks.items():
    exists = Path(path).exists() if path else False
    print(f"  {'✅' if exists else '❌'}  {name}")
    if exists: passed += 1

print(f"\n  {passed}/{len(checks)} checks passed")

# Verify exported encoder loads correctly
if LEAF_JEPA_CHECKPOINT and Path(LEAF_JEPA_CHECKPOINT).exists():
    import timm
    ckpt = torch.load(LEAF_JEPA_CHECKPOINT, map_location="cpu")
    encoder = timm.create_model(VIT_MODEL_NAME, pretrained=False, num_classes=0, global_pool="")
    missing, unexpected = encoder.load_state_dict(ckpt["target_encoder"], strict=False)
    print(f"\n  Leaf-JEPA encoder load check:")
    print(f"    Missing keys:    {len(missing)}")
    print(f"    Unexpected keys: {len(unexpected)}")
    print(f"    LP val Macro-F1: {ckpt.get('lp_val_macro_f1', 'N/A')}")
    print(f"    Best epoch:      {ckpt.get('epoch', 'N/A')}")
    print("\n  ✅ Stage 4 COMPLETE — update LEAF_JEPA_CHECKPOINT in config_stage3.py")
    print(f"     LEAF_JEPA_CHECKPOINT = Path('{LEAF_JEPA_CHECKPOINT}')")
